# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2
...,...,...
45610,"@user \""""So amazing to have the beautiful Lady...",2
45611,"9 September has arrived, which means Apple's n...",2
45612,Leeds 1-1 Sheff Wed. Giuseppe Bellusci securin...,2
45613,@user no I'm in hilton head till the 8th lol g...,1


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multiclass, output_dim = number_of_classes
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, hidden vectors are doubled in size
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1) Embedding lookup
        embedded = self.embedding(input_ids)
        # 2) LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3) Extract final hidden state
        if self.lstm.bidirectional:
            # concatenate forward & backward final hidden states
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4) Dropout
        hidden = self.dropout(hidden)

        # 5) Fully connected layer -> logits of shape [batch_size, output_dim]
        output = self.fc(hidden)
        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = test_df['label'].nunique()
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_classes]
        logits = model(input_ids)
        # CrossEntropyLoss expects [batch_size, num_classes] vs. [batch_size] labels
        loss = criterion(logits, labels)

        # Backprop and optimize
        loss.backward()
        optimizer.step()

        # Track loss
        losses.append(loss.item())

        # Convert logits -> predicted classes
        preds_cls = torch.argmax(logits, dim=1)

        # Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # Accumulate predictions and labels for metric calculations
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate average loss and overall accuracy
    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    # Calculate macro metrics
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Calculate per-class F1-scores
    # This will return a NumPy array of length = num_classes
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class

def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds_cls = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class


# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        
        val_acc, val_loss, val_prec, val_rec, val_f1_macro, val_f1_per_class = eval_model(
            model, val_loader, criterion, device
        )
        
        # Print macro stats
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}")
        
        # Print per-class F1 for train
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}")
        
        # Print per-class F1 for val
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    # Return the final metrics from the last epoch, if you like
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class
    )


In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=[
    'seed', 
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class = retval

    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,  test_f1_per_class,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.9100, Accuracy: 0.5723, Precision(macro): 0.5158, Recall(macro): 0.4537, F1(macro): 0.4311
F1 Per Class (Train): [0.06625578 0.64312928 0.58398631]
Val Loss: 0.8552, Accuracy: 0.6000, Precision(macro): 0.5558, Recall(macro): 0.5217, F1(macro): 0.5274
F1 Per Class (Val):   [0.29645094 0.6200318  0.66585067]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.7613, Accuracy: 0.6508, Precision(macro): 0.6210, Recall(macro): 0.5822, F1(macro): 0.5933
F1 Per Class (Train): [0.40356592 0.67916379 0.69731867]
Val Loss: 0.7788, Accuracy: 0.6465, Precision(macro): 0.6312, Recall(macro): 0.6038, F1(macro): 0.6090
F1 Per Class (Val):   [0.47635135 0.68016194 0.67039106]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.6494, Accuracy: 0.7108, Precision(macro): 0.6902, Recall(macro): 0.6752, F1(macro): 0.6817
F1 Per Class (Train): [0.57157692 0.72046942 0.7529094 ]
Val Loss: 0.7587, Accuracy: 0.6740, Precision(macro): 0.6461, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_10440\2112283703.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.9112, Accuracy: 0.5689, Precision(macro): 0.5159, Recall(macro): 0.4547, F1(macro): 0.4356
F1 Per Class (Train): [0.08442211 0.63447315 0.58796901]
Val Loss: 0.8524, Accuracy: 0.6155, Precision(macro): 0.6035, Recall(macro): 0.5172, F1(macro): 0.5177
F1 Per Class (Val):   [0.243309   0.67100372 0.6388309 ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.7532, Accuracy: 0.6579, Precision(macro): 0.6302, Recall(macro): 0.5939, F1(macro): 0.6053
F1 Per Class (Train): [0.42947156 0.68519724 0.70116132]
Val Loss: 0.7695, Accuracy: 0.6585, Precision(macro): 0.6281, Recall(macro): 0.6170, F1(macro): 0.6218
F1 Per Class (Val):   [0.48888889 0.66209262 0.71428571]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.6359, Accuracy: 0.7220, Precision(macro): 0.7053, Recall(macro): 0.6828, F1(macro): 0.6922
F1 Per Class (Train): [0.58339074 0.73246185 0.76081085]
Val Loss: 0.7358, Accuracy: 0.6850, Precision(macro): 0.6636, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.9117, Accuracy: 0.5661, Precision(macro): 0.5197, Recall(macro): 0.4530, F1(macro): 0.4362
F1 Per Class (Train): [0.09944479 0.63975616 0.56946115]
Val Loss: 0.8526, Accuracy: 0.6025, Precision(macro): 0.5633, Recall(macro): 0.4879, F1(macro): 0.4726
F1 Per Class (Val):   [0.1160221  0.65116279 0.65053763]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.7610, Accuracy: 0.6499, Precision(macro): 0.6181, Recall(macro): 0.5863, F1(macro): 0.5965
F1 Per Class (Train): [0.41690094 0.68051505 0.69199606]
Val Loss: 0.7926, Accuracy: 0.6520, Precision(macro): 0.6380, Recall(macro): 0.6125, F1(macro): 0.6182
F1 Per Class (Val):   [0.49411765 0.67553736 0.6850448 ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.6509, Accuracy: 0.7099, Precision(macro): 0.6893, Recall(macro): 0.6756, F1(macro): 0.6816
F1 Per Class (Train): [0.57593484 0.7213491  0.74758778]
Val Loss: 0.7466, Accuracy: 0.6720, Precision(macro): 0.6732, 

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_multiclass1.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,val_loss,val_acc,val_prec,...,test_prec,test_rec,test_f1,test_f1_per_class,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.437324,0.820476,0.810710,0.805465,0.807954,"[0.7572427572427572, 0.8175683731133313, 0.849...",0.860447,0.6660,0.645417,...,0.573342,0.564211,0.566377,"[0.5274816558216807, 0.6354797577303156, 0.536...",1233.132812,231.636230,124.436584,1233.277344,195.594238,4.127526
1,3,0.422796,0.829376,0.821013,0.815191,0.817981,"[0.7723565603090571, 0.8276801070131855, 0.853...",0.818311,0.6935,0.669630,...,0.578262,0.579863,0.575303,"[0.5440022421524664, 0.6287390985410384, 0.553...",1248.457031,232.622559,120.850600,1248.472656,195.343262,4.060030
2,5,0.449190,0.814009,0.804113,0.799733,0.801827,"[0.7528490028490028, 0.8122909299436109, 0.840...",0.826063,0.6745,0.648039,...,0.584059,0.568169,0.567607,"[0.5143458014120474, 0.6438918583933689, 0.544...",1233.644531,231.120605,127.158381,1233.652344,194.163574,4.131855


In [12]:
torch.save(model.state_dict(), 'results/lstm_multiclass1.pth')